# Intelligent Hospital Workflow Assistant
## Deep Learning Project — Milestone 2

This notebook demonstrates the full implementation of the system designed in Milestone 1.
It is organized in five phases:

1. **Setup** — environment, database, seed data
2. **Tool Demos** — each of the 5 agent tools tested standalone
3. **Agent Demo** — full ReAct loop across multiple patient scenarios
4. **FastAPI Backend** — starting the server and testing endpoints
5. **Evaluation** — priority assignment accuracy across test cases

---
## Phase 1 — Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set — edit .env first'
print('API key loaded.')

In [ ]:
from backend.database import engine
from backend.models import Base
from backend.seed_data import seed

Base.metadata.create_all(bind=engine)
seed()

In [ ]:
from backend.database import get_session
from backend.models import Doctor

db = get_session()
doctors = db.query(Doctor).all()
db.close()

print(f'{len(doctors)} doctors seeded:')
for d in doctors:
    print(f'  [{d.id}] {d.name} — {d.specialty} ({d.department})')

In [ ]:
from backend.models import User

db = get_session()
for email, age, history in [
    ('alice@example.com', 45, 'Type 2 diabetes, hypertension'),
    ('bob@example.com', 28, 'None'),
]:
    if not db.query(User).filter(User.email == email).first():
        db.add(User(email=email, age=age, chronic_disease_history=history))
db.commit()
db.close()
print('Test patients registered.')

---
## Phase 2 — Tool Demos

Each tool is tested in isolation before being given to the agent.

In [ ]:
from backend.tools import (
    get_patient_history,
    find_available_doctor,
    book_appointment,
    save_medical_report,
    send_confirmation_email,
)
import json

def pretty(result):
    print(json.dumps(result, indent=2, default=str))

In [ ]:
print('=== Tool 1: get_patient_history ===')
pretty(get_patient_history.invoke({'email': 'alice@example.com'}))

In [ ]:
print('=== Tool 2: find_available_doctor (cardiology) ===')
doctor_result = find_available_doctor.invoke({'specialty': 'cardiology'})
pretty(doctor_result)

In [ ]:
print('=== Tool 3: book_appointment ===')
d = doctor_result['data']
booking = book_appointment.invoke({
    'user_email': 'alice@example.com',
    'doctor_id': d['doctor_id'],
    'slot_id': d['slot_id'],
    'priority': 1,
})
pretty(booking)

In [ ]:
print('=== Tool 4: save_medical_report ===')
appt_id = booking['data']['appointment_id']
pretty(save_medical_report.invoke({
    'appointment_id': appt_id,
    'summary': 'Patient reports persistent chest tightness for 3 days. Known hypertension. Triage: Priority 1.',
    'medication_recommendations': 'Continue antihypertensive medication. Avoid strenuous activity.',
}))

In [ ]:
print('=== Tool 5: send_confirmation_email (logged, no SMTP configured) ===')
pretty(send_confirmation_email.invoke({
    'to_email': 'alice@example.com',
    'doctor_name': d['doctor_name'],
    'slot_datetime': d['slot_datetime'],
    'department': d['department'],
}))

---
## Phase 3 — Agent Demo

The LangChain ReAct agent runs the full intake flow end-to-end.
Watch the **Thought → Action → Observation** trace in the output.

In [ ]:
from backend.agent import build_agent_executor

executor = build_agent_executor()

In [ ]:
print('='*60)
print('SCENARIO 1: Urgent — chest pain + shortness of breath (Priority 0)')
print('='*60)

r = executor.invoke({
    'input': 'Patient email: bob@example.com\n\nPatient message: I have severe chest pain radiating to my left arm and I can barely breathe. It started 20 minutes ago.'
})
print('\n--- AGENT FINAL RESPONSE ---')
print(r['output'])

In [ ]:
print('='*60)
print('SCENARIO 2: Multi-turn — vague symptoms requiring follow-up')
print('='*60)

executor2 = build_agent_executor()

r1 = executor2.invoke({'input': 'Patient email: alice@example.com\n\nPatient message: I have been feeling dizzy and tired lately.'})
print('[AGENT TURN 1]:', r1['output'])

r2 = executor2.invoke({'input': 'Patient email: alice@example.com\n\nPatient message: About 5 days. The dizziness is worst in the morning and I also have a headache.'})
print('\n[AGENT TURN 2]:', r2['output'])

In [ ]:
print('='*60)
print('SCENARIO 3: Routine — prescription renewal (Priority 3)')
print('='*60)

executor3 = build_agent_executor()
r = executor3.invoke({'input': 'Patient email: bob@example.com\n\nPatient message: I need to renew my asthma inhaler prescription. No new symptoms, just running out.'})
print('[AGENT RESPONSE]:', r['output'])

---
## Phase 4 — FastAPI Backend

In [ ]:
import subprocess, time

server = subprocess.Popen(
    ['python3', '-m', 'uvicorn', 'backend.main:app', '--host', '127.0.0.1', '--port', '8000'],
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print('Server started. PID:', server.pid)

In [ ]:
import requests
BASE = 'http://127.0.0.1:8000'

print('Health:', requests.get(f'{BASE}/health').json())
print('Identify alice:', requests.post(f'{BASE}/identify', json={'email': 'alice@example.com'}).json())
print('Identify unknown:', requests.post(f'{BASE}/identify', json={'email': 'new@example.com'}).json())

In [ ]:
resp = requests.post(f'{BASE}/register', json={
    'email': 'new@example.com',
    'age': 35,
    'chronic_disease_history': 'Asthma',
})
print('Register:', resp.json())

In [ ]:
resp = requests.post(f'{BASE}/chat', json={
    'email': 'new@example.com',
    'message': 'I have had a skin rash on my arm for a week. It is itchy and slightly red.',
})
data = resp.json()
print('Session ID:', data['session_id'])
print('\nAgent Response:\n', data['response'])

In [ ]:
server.terminate()
print('Server stopped.')

---
## Phase 5 — Evaluation

Priority assignment accuracy across labelled test cases.

In [ ]:
import re

TEST_CASES = [
    ('Severe chest pain radiating to left arm, started 10 minutes ago', 0),
    ('Sudden loss of speech and facial drooping on the right side', 0),
    ('High fever 39.5 degrees for two days with severe headache', 1),
    ('Moderate knee pain after a fall, can still walk', 2),
    ('Dry cough and mild fatigue for the past week', 2),
    ('Routine blood pressure check, no symptoms', 3),
    ('Prescription renewal for daily metformin', 3),
]

results = []
for desc, expected in TEST_CASES:
    ex = build_agent_executor()
    resp = ex.invoke({'input': f'Patient email: bob@example.com\n\nPatient message: {desc}'})
    output = resp['output']
    match = re.search(r'[Pp]riority\s*[:\-]?\s*(\d)', output)
    predicted = int(match.group(1)) if match else -1
    correct = (predicted == expected)
    results.append((desc[:55], expected, predicted, correct))
    print(f'[{"PASS" if correct else "FAIL"}] Expected P{expected}, Got P{predicted} | {desc[:55]}')

accuracy = sum(r[3] for r in results) / len(results)
print(f'\nAccuracy: {accuracy:.0%} ({sum(r[3] for r in results)}/{len(results)})')

---
## Summary

| Component | Status |
|---|---|
| PostgreSQL database (6 tables) | Done |
| SQLAlchemy ORM models | Done |
| Doctor + slot seeding (8 doctors, 10 slots each) | Done |
| `get_patient_history` tool | Done |
| `find_available_doctor` tool | Done |
| `book_appointment` tool (atomic, no double-booking) | Done |
| `save_medical_report` tool | Done |
| `send_confirmation_email` tool | Done |
| LangChain ReAct agent (Claude claude-sonnet-4-6) | Done |
| FastAPI backend (4 endpoints) | Done |
| Multi-turn conversation memory | Done |
| Conversation audit trail in DB | Done |

All agent outputs are framed as preliminary recommendations pending physician review.